# Tiny Shakespeare
Load the dataset into `text` and preview its opening.

In [1]:
from pathlib import Path

# Supports launching Jupyter from either the repo root or src/.
text_path = Path("input.txt")
if not text_path.exists():
    text_path = Path("src/input.txt")

text = text_path.read_text(encoding="utf-8")
print(f"Loaded {len(text):,} characters")
print(text[:500])


Loaded 1,115,394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [2]:
chars = sorted(set(text))
print(chars)
print(f"Unique characters: {len(chars)}")


['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Unique characters: 65


In [3]:
# Each character gets one token ID, following the sorted chars list.
stoi = {char: i for i, char in enumerate(chars)}
itos = {i: char for i, char in enumerate(chars)}


def encode(string):
    return [stoi[char] for char in string]


def decode(tokens):
    return "".join(itos[token] for token in tokens)


sample = "Hello, world!"
tokens = encode(sample)
print("Token IDs:", tokens)
print("Decoded:", decode(tokens))
assert decode(tokens) == sample


Token IDs: [20, 43, 50, 50, 53, 6, 1, 61, 53, 56, 50, 42, 2]
Decoded: Hello, world!


In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print("Shape:", data.shape)
print("Dtype:", data.dtype)
print("First 100 token IDs:", data[:100])
assert decode(data.tolist()) == text


[local path redacted] UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at [local path redacted]
  cpu = _conversion_method_template(device=torch.device("cpu"))


Shape: torch.Size([1115394])
Dtype: torch.int64
First 100 token IDs: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [5]:
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

print(f"Train tokens: {len(train_data):,}")
print(f"Test tokens: {len(test_data):,}")



Train tokens: 1,003,854
Test tokens: 111,540


In [6]:

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"When input is {context}, the target: {target}")
    print(f"  {decode(context.tolist())!r} -> {decode([target.item()])!r}")


When input is tensor([18]), the target: 47
  'F' -> 'i'
When input is tensor([18, 47]), the target: 56
  'Fi' -> 'r'
When input is tensor([18, 47, 56]), the target: 57
  'Fir' -> 's'
When input is tensor([18, 47, 56, 57]), the target: 58
  'Firs' -> 't'
When input is tensor([18, 47, 56, 57, 58]), the target: 1
  'First' -> ' '
When input is tensor([18, 47, 56, 57, 58,  1]), the target: 15
  'First ' -> 'C'
When input is tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
  'First C' -> 'i'
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58
  'First Ci' -> 't'


In [7]:
torch.manual_seed(1337)
batch_size = 32
block_size = 8


def get_batch(split):
    data = train_data if split == "train" else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y


xb, yb = get_batch("train")
print("Inputs:", xb.shape, xb)
print("Targets:", yb.shape, yb)


Inputs: torch.Size([32, 8]) tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46],
        [43,  1, 51, 39, 63,  1, 40, 43],
        [58, 46, 43,  1, 43, 39, 56, 57],
        [39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1],
        [39, 51,  1, 39, 44, 56, 39, 47],
        [17, 24, 21, 38, 13, 14, 17, 32],
        [ 1, 39, 52, 42,  1, 45, 43, 50],
        [ 1, 58, 46, 39, 58,  1, 42, 53],
        [ 1, 61, 53, 59, 50, 42,  1, 21],
        [59, 57, 40, 39, 52, 42,  1, 40],
        [52, 42,  8,  0,  0, 23, 21, 26],
        [45, 53, 42, 57,  0, 23, 43, 43],
        [52,  1, 61, 39, 57,  1, 51, 53],
      

In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
vocab_size = len(chars)
n_embedd = 32


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embedd):
        super().__init__()
        # One 32-feature vector per token; project features to vocabulary scores.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embedd)
        self.lm_head = nn.Linear(n_embedd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets: (B, T); logits: (B, T, C).
        token_embeddings = self.token_embedding_table(idx)  # (B, T, 32)
        logits = self.lm_head(token_embeddings)  # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx: (B, T); append one sampled token per sequence each step.
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # Last position: (B, C).
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


m = BigramLanguageModel(vocab_size, n_embedd)
logits, loss = m(xb, yb)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=100)
print("Generated shape:", generated.shape)
print(decode(generated[0].tolist()))


Logits shape: torch.Size([32, 8, 65])
Loss: 4.3511199951171875
Generated shape: torch.Size([1, 101])

hYQRnbbmkMTUwbiu$?3KHvybsMEEFNLyb!SZgyGzRX$oNqTs!roUNLjMXM!EjT!hjmfH'ER3cOn.kvgAuau&e;m-CNLkfMW HT'R


In [9]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()

max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step == 0 or (step + 1) % 500 == 0 or step + 1 == max_iters:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")


Step 1: loss = 4.3955
Step 500: loss = 2.6869
Step 1000: loss = 2.5488
Step 1500: loss = 2.4488
Step 2000: loss = 2.5055
Step 2500: loss = 2.4174
Step 3000: loss = 2.3898
Step 3500: loss = 2.6033
Step 4000: loss = 2.4817
Step 4500: loss = 2.4913
Step 5000: loss = 2.4889


In [10]:
eval_iters = 200


@torch.no_grad()
def estimate_loss():
    out = {}
    was_training = m.training
    m.eval()
    try:
        for split in ("train", "test"):
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split)
                _, loss = m(xb, yb)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
    finally:
        m.train(was_training)
    return out


estimated_losses = estimate_loss()
print(estimated_losses)

baseline_losses = estimated_losses.copy()
baseline_model = m


{'train': 2.4770517349243164, 'test': 2.493652582168579}


In [11]:
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



ANSabe shind t Oncrito my.
d
Pe the pita'nou, mKIthithire shothou hethakearierve flase ait at an l wimathe lde
E gofo Cumugat he, wisha at:
WAf bun, mor atawearorellernounove f an:
I orivedifod nt ashathingrothe
Durgathe RILLpond heplge-d re ome angeavifoforispatowne oundmicked;
Yord'sepvond, y
Tharst t he g, ctad,
Bud it thowow m, t nd f made; al t?
Tely horishoothe Y alo, isee; strgnouse ty CASTowinthito ther fr ait ll isthe thend nowou wbl hyor ha seice yeryefu y hitis gand s th,

:
HEOfal mn


## Add position embeddings
Baseline results above remain separate. Train a fresh position-aware model below.

In [12]:
class PositionalLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embedd):
        super().__init__()
        # One 32-feature vector per token; project features to vocabulary scores.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embedd)
        self.position_embedding_table = nn.Embedding(block_size, n_embedd)
        self.lm_head = nn.Linear(n_embedd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets: (B, T); logits: (B, T, C).
        token_embeddings = self.token_embedding_table(idx)  # (B, T, 32)
        _, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)  # (T, 32)
        x = token_embeddings + position_embeddings  # Broadcast across batch.
        logits = self.lm_head(x)  # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx: (B, T); append one sampled token per sequence each step.
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]  # Last position: (B, C).
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [13]:
# Fresh model: same embedding width and training settings, plus positions.
torch.manual_seed(1337)
m = PositionalLanguageModel(vocab_size, n_embedd)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()

max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step == 0 or (step + 1) % 500 == 0 or step + 1 == max_iters:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

position_losses = estimate_loss()
print("\nLoss comparison (200 batches per split):")
print(f"Token only:       train={baseline_losses['train']:.4f}, test={baseline_losses['test']:.4f}")
print(f"Token + position: train={position_losses['train']:.4f}, test={position_losses['test']:.4f}")


Step 1: loss = 4.5150
Step 500: loss = 2.6772
Step 1000: loss = 2.5814
Step 1500: loss = 2.4210
Step 2000: loss = 2.4975
Step 2500: loss = 2.4261
Step 3000: loss = 2.5030
Step 3500: loss = 2.5029
Step 4000: loss = 2.5205
Step 4500: loss = 2.5525
Step 5000: loss = 2.4868

Loss comparison (200 batches per split):
Token only:       train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880


In [14]:
# Toy example: batch, time positions, channels.
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(x.shape)


torch.Size([4, 8, 2])


In [15]:
# Causal bag-of-words: average positions 0 through t, channel by channel.
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]  # (t + 1, C), including current position.
        xbow[b, t] = torch.mean(xprev, dim=0)

print("First sequence input:")
print(x[0])
print("First sequence running averages:")
print(xbow[0])


First sequence input:
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
First sequence running averages:
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [16]:
# Each row averages only current and earlier positions.
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(dim=1, keepdim=True)
xbow_matmul = wei @ x  # (T, T) @ (B, T, C) -> (B, T, C)

print("Averaging weights:")
print(wei)
print("Output shape:", xbow_matmul.shape)
print("Matches loop:", torch.allclose(xbow_matmul, xbow))


Averaging weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
Output shape: torch.Size([4, 8, 2])
Matches loop: True


In [35]:
print(xbow_matmul[0])


tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [18]:
# Same averaging via masked_fill + softmax: future scores -> -inf -> weight 0.
tril = torch.tril(torch.ones(T, T))
wei_softmax = torch.zeros(T, T)  # Equal scores for every visible position.
wei_softmax = wei_softmax.masked_fill(tril == 0, float("-inf"))
print("Masked scores:")
print(wei_softmax)

wei_softmax = F.softmax(wei_softmax, dim=-1)
xbow_softmax = wei_softmax @ x

print("Softmax weights:")
print(wei_softmax)
print("Matches tril/sum weights:", torch.allclose(wei_softmax, wei))
print("Matches loop:", torch.allclose(xbow_softmax, xbow))

Masked scores:
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
Softmax weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 

In [19]:
# One self-attention head on toy data: scores now come from the tokens.
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x_att = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)    # "what I offer"
query = nn.Linear(C, head_size, bias=False)  # "what I'm looking for"
value = nn.Linear(C, head_size, bias=False)  # "what I pass on if chosen"

k = key(x_att)    # (B, T, 16)
q = query(x_att)  # (B, T, 16)
v = value(x_att)  # (B, T, 16)

# score[b, i, j] = q_i . k_j, scaled like Module 2.1.
scores = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T)
tril = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(tril == 0, float("-inf"))
att_wei = F.softmax(scores, dim=-1)  # (B, T, T)
out = att_wei @ v  # (B, T, 16)

print("Attention weights, first sequence:")
print(att_wei[0].round(decimals=2))
print("Output shape:", out.shape)


Attention weights, first sequence:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3100, 0.2900, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.2200, 0.2400, 0.2100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1500, 0.2000, 0.1700, 0.1500, 0.3400, 0.0000, 0.0000, 0.0000],
        [0.1300, 0.2500, 0.1300, 0.1100, 0.3100, 0.0700, 0.0000, 0.0000],
        [0.1600, 0.2000, 0.1100, 0.1100, 0.1400, 0.1700, 0.1100, 0.0000],
        [0.0800, 0.1200, 0.1100, 0.1500, 0.1100, 0.1100, 0.1600, 0.1600]],
       grad_fn=<RoundBackward1>)
Output shape: torch.Size([4, 8, 16])


## Model with causal averaging
Average token-plus-position vectors before the output layer. Earlier model cells remain unchanged.

In [20]:
class CausalAverageLanguageModel(PositionalLanguageModel):
    # Reuse embedding tables, output layer, and generation from position model.
    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)
        x = token_embeddings + position_embeddings  # (B, T, n_embedd)

        # Equal scores, future masked to -inf; softmax turns them into averages.
        tril = torch.tril(torch.ones(T, T, device=x.device))
        wei = torch.zeros(T, T, device=x.device, dtype=x.dtype)
        wei = wei.masked_fill(tril == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)  # (T, T)
        xbow_matmul = wei @ x  # (B, T, n_embedd)
        logits = self.lm_head(xbow_matmul)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            C = logits.shape[-1]
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


In [21]:
# Train a fresh model with causal averaging; preserve earlier comparisons.
torch.manual_seed(1337)
m = CausalAverageLanguageModel(vocab_size, n_embedd)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

average_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Step 1: loss = 4.2683
Step 500: loss = 2.9804
Step 1000: loss = 2.9372
Step 1500: loss = 2.8320
Step 2000: loss = 2.8795
Step 2500: loss = 2.8479
Step 3000: loss = 2.8796
Step 3500: loss = 2.8554
Step 4000: loss = 2.8648
Step 4500: loss = 2.9992
Step 5000: loss = 2.9700

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140


In [22]:
# Qualitative sample from the trained causal-average model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



EINN,NCT  LeaEI
vBleleowen ie'dsh d ldane  nr t chaemetanl
nyat
reBreidn
eTtteih n ee gporasre. 
ae
U Fryrbr o, edleds, n miyfdAg, 
e rkhiae g WtMeforl d ewelof q woLrlemidl
g, oahntuf, lWy  icllnylelid ds fa Gilrhes!w
Tiu
iefrEdI  h hlaymeg d  thoOmad  yynWa onn tpmghee ehmcayli al,eW lhofupetco lE  aitmems e tihei hl oero wan rmiyis tgs ohlenyrs ihss omur
Ipy tC idto amwisu t yhI rau m hetraaelievf
meis ercfl,hr  boseed,b
 ono npF hnoyf oord se e nhmgre,i oo
mr anenu at ggseuhti lne yh lt aou 


## Model with one self-attention head
Replace equal averaging scores with learned scores `q @ k.T`. Earlier models remain unchanged.

In [23]:
class Head(nn.Module):
    # One causal self-attention head: same steps as the toy cell above.
    def __init__(self, n_embedd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embedd, head_size, bias=False)
        self.query = nn.Linear(n_embedd, head_size, bias=False)
        self.value = nn.Linear(n_embedd, head_size, bias=False)
        # Fixed mask, not learned; register_buffer keeps it out of parameters.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        _, T, _ = x.shape
        k = self.key(x)  # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)
        d_k = k.shape[-1]
        scores = q @ k.transpose(-2, -1) / d_k**0.5  # (B, T, T), / sqrt(d_k)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(scores, dim=-1)  # (B, T, T)
        return wei @ v  # (B, T, head_size)


class SelfAttentionLanguageModel(PositionalLanguageModel):
    # Reuse embedding tables, output layer, and generation from position model.
    def __init__(self, vocab_size, n_embedd):
        super().__init__(vocab_size, n_embedd)
        # head_size = n_embedd so the output still fits lm_head (32 -> 65).
        self.sa_head = Head(n_embedd, n_embedd)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)
        x = token_embeddings + position_embeddings  # (B, T, n_embedd)

        x = self.sa_head(x)  # Learned weights replace equal averaging.
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            C = logits.shape[-1]
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


In [24]:
# Train a fresh self-attention model; same settings as earlier models.
torch.manual_seed(1337)
m = SelfAttentionLanguageModel(vocab_size, n_embedd)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

attention_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Step 1: loss = 4.2544
Step 500: loss = 2.8124
Step 1000: loss = 2.5106
Step 1500: loss = 2.5600
Step 2000: loss = 2.4305
Step 2500: loss = 2.3753
Step 3000: loss = 2.3976
Step 3500: loss = 2.4126
Step 4000: loss = 2.4243
Step 4500: loss = 2.4296
Step 5000: loss = 2.2673

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100


In [25]:
# Qualitative sample from the trained self-attention model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



Hicuso thel he than ouds macifrcniond wiloledl, ome my INGcaln of youre slans,
Burs:
A reo nis. I ak,
N po the! lwigouru kidad,
Thalivit the evere:

NORII: ber, ge. O
Giond;
TIresemen, ean, cocharwe, le ce ss lllofer le en omeyod thuun wathave:
Afinc'l
Aur, resu de'd,
A:

Fycary thum:
We t,
I, rillforlolowe,, whel you anonser,
Thor.
Caikn at:
n
I tha in dengtooud if pay wer, yost how!

Theves; hans we il idoduf, ans ad thart, hand sto hin:
Mrouit.


RY IFor, nist krcay micoour bes, thicavey bant


## Model with multi-head self-attention
Four heads of size 8 run in parallel; outputs are concatenated back to 32 features. Earlier models remain unchanged.

In [26]:
class MultiHeadAttention(nn.Module):
    # Several independent heads; each learns its own attention pattern.
    def __init__(self, n_embedd, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embedd, head_size) for _ in range(num_heads)]
        )

    def forward(self, x):
        # Each head: (B, T, head_size); concatenate on features.
        return torch.cat([h(x) for h in self.heads], dim=-1)


class MultiHeadLanguageModel(PositionalLanguageModel):
    # Same as SelfAttentionLanguageModel, with 4 heads of size 8.
    def __init__(self, vocab_size, n_embedd, num_heads=4):
        super().__init__(vocab_size, n_embedd)
        # 4 heads x 8 features = 32, so output still fits lm_head.
        self.sa_heads = MultiHeadAttention(
            n_embedd, num_heads, n_embedd // num_heads
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)
        x = token_embeddings + position_embeddings  # (B, T, n_embedd)

        x = self.sa_heads(x)  # (B, T, n_embedd)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            C = logits.shape[-1]
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


In [27]:
# Train a fresh multi-head model; same settings as earlier models.
torch.manual_seed(1337)
m = MultiHeadLanguageModel(vocab_size, n_embedd)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

multihead_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
    ("Multi-head (4 x 8)", multihead_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Step 1: loss = 4.2510
Step 500: loss = 2.7614
Step 1000: loss = 2.4481
Step 1500: loss = 2.4452
Step 2000: loss = 2.3483
Step 2500: loss = 2.2866
Step 3000: loss = 2.3628
Step 3500: loss = 2.2881
Step 4000: loss = 2.3526
Step 4500: loss = 2.3101
Step 5000: loss = 2.0638

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100
Multi-head (4 x 8): train=2.2294, test=2.2589


In [28]:
# Qualitative sample from the trained multi-head model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



Hick, of in thy land;
Gomeace ron,
O yobloledl, ome my thectler acy adinglans,
Bore:
A feown! heur tons po the! lof hey Rknals,
Thall dout, tevere:

Non Iy bertst
os tile
Whtuldesembenee to cochur kod,
Tce shilll ferche en ome:
d thiun wath ma:
ading'l
And, reid drim,
And
Fy
ORIUSE:
Cidser,
I, rive!
Whou wer, wheld bumanenser,
Thar.
Caike at:
nd your in dengtoy don's the shat sst howne halll'll wis we it it dif'll:
bad that Yo sor sto hin:
y heit.

QUCEIF:
G I stik cosevickour his:, ovinthy bant


## Model with one transformer block (attention + feed-forward)
Attention lets tokens gather information from earlier tokens; feed-forward lets each token process it. No residuals or LayerNorm yet.

In [29]:
class FeedForward(nn.Module):
    # Per-token MLP: expand to 4x width, ReLU, project back. No mixing across tokens.
    def __init__(self, n_embedd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embedd, 4 * n_embedd),  # 32 -> 128
            nn.ReLU(),
            nn.Linear(4 * n_embedd, n_embedd),  # 128 -> 32
        )

    def forward(self, x):
        return self.net(x)  # (B, T, n_embedd)


class Block(nn.Module):
    # Communication (attention) followed by computation (feed-forward).
    def __init__(self, n_embedd, num_heads):
        super().__init__()
        self.sa_heads = MultiHeadAttention(
            n_embedd, num_heads, n_embedd // num_heads
        )
        self.ffwd = FeedForward(n_embedd)

    def forward(self, x):
        x = self.sa_heads(x)  # (B, T, n_embedd)
        x = self.ffwd(x)  # (B, T, n_embedd)
        return x


class BlockLanguageModel(PositionalLanguageModel):
    # Multi-head model plus a feed-forward layer, wrapped as one block.
    def __init__(self, vocab_size, n_embedd, num_heads=4):
        super().__init__(vocab_size, n_embedd)
        self.block = Block(n_embedd, num_heads)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)
        x = token_embeddings + position_embeddings  # (B, T, n_embedd)

        x = self.block(x)  # (B, T, n_embedd)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            C = logits.shape[-1]
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


In [30]:
# Train a fresh block model; same settings as earlier models.
torch.manual_seed(1337)
m = BlockLanguageModel(vocab_size, n_embedd)
print("Parameters:", sum(p.numel() for p in m.parameters()))
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

block_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
    ("Multi-head (4 x 8)", multihead_losses),
    ("Block (multi-head + FFN)", block_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Parameters: 15905
Step 1: loss = 4.2020
Step 500: loss = 2.6965
Step 1000: loss = 2.4459
Step 1500: loss = 2.3198
Step 2000: loss = 2.3497
Step 2500: loss = 2.2091
Step 3000: loss = 2.1395
Step 3500: loss = 2.0853
Step 4000: loss = 2.0384
Step 4500: loss = 2.1882
Step 5000: loss = 2.1990

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100
Multi-head (4 x 8): train=2.2294, test=2.2589
Block (multi-head + FFN): train=2.1505, test=2.1888


In [31]:
# Qualitative sample from the trained block model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



MELNUT:
I worhate, the be the my'l duncooud frosughty ie Umake all harld your athuelt or 't visnedseck, a thates benceniguet pon:
Angus me Mass, a the, orly.


Whill to, carvenf urrom'dpenight thament and on wet resth eat my, this bremy but thuque to the too schifish son.

Ano yell budair to the tiy! peeak it binn.

FANURCER:
Whou thacke tand band?

LOELTILOUTENIUTI:
HhENSORD Cis vesis to cork dry chou gridfe, clike, up ring, it forse lath don,
Fillall a rearll ospordop of hath thourds id to ee 


## Model with three stacked blocks
Same `Block` repeated three times in sequence. Still no residuals or LayerNorm.

In [32]:
class StackedBlockLanguageModel(PositionalLanguageModel):
    # Three blocks in a row: output of one block is input to the next.
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3):
        super().__init__(vocab_size, n_embedd)
        self.blocks = nn.Sequential(
            *[Block(n_embedd, num_heads) for _ in range(n_layer)]
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        positions = torch.arange(T, device=idx.device)
        position_embeddings = self.position_embedding_table(positions)
        x = token_embeddings + position_embeddings  # (B, T, n_embedd)

        x = self.blocks(x)  # block 1 -> block 2 -> block 3, (B, T, n_embedd)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            C = logits.shape[-1]
            loss = F.cross_entropy(
                logits.reshape(B * T, C), targets.reshape(B * T)
            )
        return logits, loss


In [33]:
# Train a fresh 3-block model; same settings as earlier models.
torch.manual_seed(1337)
m = StackedBlockLanguageModel(vocab_size, n_embedd)
print("Parameters:", sum(p.numel() for p in m.parameters()))
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

stacked_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
    ("Multi-head (4 x 8)", multihead_losses),
    ("Block (multi-head + FFN)", block_losses),
    ("3 blocks, no residual", stacked_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Parameters: 38753
Step 1: loss = 4.1695
Step 500: loss = 3.1429
Step 1000: loss = 2.4848
Step 1500: loss = 2.5965
Step 2000: loss = 2.4788
Step 2500: loss = 2.2892
Step 3000: loss = 2.1764
Step 3500: loss = 2.4026
Step 4000: loss = 2.3814
Step 4500: loss = 2.3665
Step 5000: loss = 2.2007

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100
Multi-head (4 x 8): train=2.2294, test=2.2589
Block (multi-head + FFN): train=2.1505, test=2.1888
3 blocks, no residual: train=2.1972, test=2.2215


In [34]:
# Qualitative sample from the trained 3-block model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



KLA:
Af an'g
And, reiude. Bace, lutcard thum the tein, rill Cremcower, world of an nolr,
Tong.

LINGGEO:
And tof rote agtoyad ther my snnoy sting,
The lalse hans we it it dof, ans agr marll hand store they heit.

RUCIIFGC: IG to ropeever our his herich hou antergo and porltand hold stead him. Le nrow'd, the gon,
Ant me und he plat, Aut it, stery? xe fork my to morde wordes.
Lome ther spind arit and NOENENCIUS:
Sesly is shat I and to hate houne'd my mudst.

KLIANLA:
Wo way,
To sow. Theard wer mon


## Model with three residual blocks
Each layer adds its output to the running `x` (residual stream). Same layers and sizes as the 3-block model.

In [36]:
class ResidualBlock(Block):
    # Same layers as Block; each layer now adds its output to x.
    def forward(self, x):
        x = x + self.sa_heads(x)  # (B, T, n_embedd)
        x = x + self.ffwd(x)  # (B, T, n_embedd)
        return x


class ResidualStackLanguageModel(StackedBlockLanguageModel):
    # Same 3-block stack; only the block type changes.
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer)
        self.blocks = nn.Sequential(
            *[ResidualBlock(n_embedd, num_heads) for _ in range(n_layer)]
        )


In [37]:
# Train a fresh 3-block residual model; same settings as earlier models.
torch.manual_seed(1337)
m = ResidualStackLanguageModel(vocab_size, n_embedd)
print("Parameters:", sum(p.numel() for p in m.parameters()))
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

residual_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
    ("Multi-head (4 x 8)", multihead_losses),
    ("Block (multi-head + FFN)", block_losses),
    ("3 blocks, no residual", stacked_losses),
    ("3 blocks, residual", residual_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Parameters: 38753
Step 1: loss = 4.5460
Step 500: loss = 2.3548
Step 1000: loss = 2.2667
Step 1500: loss = 2.2395
Step 2000: loss = 2.1624
Step 2500: loss = 2.0744
Step 3000: loss = 1.9898
Step 3500: loss = 2.0479
Step 4000: loss = 1.9639
Step 4500: loss = 2.0365
Step 5000: loss = 1.8781

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100
Multi-head (4 x 8): train=2.2294, test=2.2589
Block (multi-head + FFN): train=2.1505, test=2.1888
3 blocks, no residual: train=2.1972, test=2.2215
3 blocks, residual: train=2.0229, test=2.0954


In [38]:
# Qualitative sample from the trained residual model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



Met you stilf honot Seys bake wrials. This a to to kids.
'Tont in sheach and him mest
Ind I do mule to arble-do'd with sornting in! with tlervier to spoldies,
Your must not Myous candery she contity;
Yout what nistop?

GLEO:
Whic,
Sese blow, vows! wort be us sond furthempalges.

KING RAMANTIY:
And your lands farlouses his him grem at.

GUtIRDY IV:
For this fome, borand,
Sothatimfarevion, tan; Mart save the not Cre unt.

HENRY Thim
Go kirs?
Wite; ore theat king eathiriends his to for them and wer


## Model with LayerNorm (pre-norm)
Same residual 3-block stack. Each layer reads a normalized copy of `x`; a final LayerNorm runs before `lm_head`.

In [40]:
# LayerNorm on one token: mean ~0, std ~1 before learned scale/shift.
ln = nn.LayerNorm(4)
v = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
out = ln(v)
print("input :", v)
print("output:", out.detach().round(decimals=3))
print("mean, std:", round(out.mean().item(), 3), round(out.std(unbiased=False).item(), 3))


input : tensor([[2., 4., 6., 8.]])
output: tensor([[-1.3420, -0.4470,  0.4470,  1.3420]])
mean, std: 0.0 1.0


In [41]:
class LayerNormBlock(ResidualBlock):
    # Residual block plus pre-norm: each layer reads a normalized copy of x.
    def __init__(self, n_embedd, num_heads):
        super().__init__(n_embedd, num_heads)
        self.ln1 = nn.LayerNorm(n_embedd)  # before attention
        self.ln2 = nn.LayerNorm(n_embedd)  # before feed-forward

    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))  # stream itself stays unnormalized
        x = x + self.ffwd(self.ln2(x))
        return x


class LayerNormStackLanguageModel(StackedBlockLanguageModel):
    # Same 3-block residual stack; LayerNorm inside blocks and before lm_head.
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer)
        self.blocks = nn.Sequential(
            *[LayerNormBlock(n_embedd, num_heads) for _ in range(n_layer)],
            nn.LayerNorm(n_embedd),  # final ln_f before lm_head
        )


In [42]:
# Train a fresh 3-block residual + LayerNorm model; same settings as earlier models.
torch.manual_seed(1337)
m = LayerNormStackLanguageModel(vocab_size, n_embedd)
print("Parameters:", sum(p.numel() for p in m.parameters()))
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step == 0 or (step + 1) % 500 == 0:
        print(f"Step {step + 1}: loss = {loss.item():.4f}")

layernorm_losses = estimate_loss()
print("\nEstimated losses (200 batches per split):")
for name, scores in (
    ("Token only", baseline_losses),
    ("Token + position", position_losses),
    ("Causal average", average_losses),
    ("Self-attention head", attention_losses),
    ("Multi-head (4 x 8)", multihead_losses),
    ("Block (multi-head + FFN)", block_losses),
    ("3 blocks, no residual", stacked_losses),
    ("3 blocks, residual", residual_losses),
    ("3 blocks, residual + LayerNorm", layernorm_losses),
):
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}")


Parameters: 39201
Step 1: loss = 4.2111
Step 500: loss = 2.4105
Step 1000: loss = 2.2731
Step 1500: loss = 2.2085
Step 2000: loss = 2.2047
Step 2500: loss = 2.0240
Step 3000: loss = 1.9854
Step 3500: loss = 2.0220
Step 4000: loss = 1.9541
Step 4500: loss = 2.0530
Step 5000: loss = 1.8369

Estimated losses (200 batches per split):
Token only: train=2.4771, test=2.4937
Token + position: train=2.4814, test=2.4880
Causal average: train=2.8184, test=2.8140
Self-attention head: train=2.3868, test=2.4100
Multi-head (4 x 8): train=2.2294, test=2.2589
Block (multi-head + FFN): train=2.1505, test=2.1888
3 blocks, no residual: train=2.1972, test=2.2215
3 blocks, residual: train=2.0229, test=2.0954
3 blocks, residual + LayerNorm: train=1.9988, test=2.0719


In [43]:
# Qualitative sample from the trained LayerNorm model.
m.eval()
context = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))



MOUSEES:
Oreffor this; say,
That worth the a to to keconcesvard, Asurcess if him mest
Indgrier: she to arn the have no sorne or hey no yould of all Godk'ding,
Yourserdern the foay, and your contice.

Yout what nifgolk'd, inbly it,
And this covows! I
No blouiess.

PRLEO:
Whight.

KINGURAM:
Them shee vipel?
Shick'd flisI is him ghem at.

GUOUCA:
Fithane!
For forgeat, and canteakinf waver Than
Our, they lead's vere, not he lowly,
The God, ress the; of are apake one this thy marred him.

DURIOTULEY:


## Experiments: longer training vs longer context
Same LayerNorm model. A doubles training steps; B quadruples `block_size`. `gap` = test − train (large gap = overfitting).

In [44]:
def train_and_eval(model, iters):
    # Same loop as earlier cells; returns estimate_loss() for this model.
    global m
    m = model
    torch.manual_seed(1337)
    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
    m.train()
    for step in range(iters):
        xb, yb = get_batch("train")
        _, loss = m(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if (step + 1) % 2500 == 0:
            print(f"Step {step + 1}: loss = {loss.item():.4f}")
    return estimate_loss()


def report(name, scores):
    gap = scores["test"] - scores["train"]
    print(f"{name}: train={scores['train']:.4f}, test={scores['test']:.4f}, gap={gap:.4f}")


In [45]:
# Experiment A: same LayerNorm model, block_size 8, twice the steps.
torch.manual_seed(1337)
longer_losses = train_and_eval(LayerNormStackLanguageModel(vocab_size, n_embedd), 10000)
print()
report("Baseline (block 8, 5000 steps)", layernorm_losses)
report("A: block 8, 10000 steps", longer_losses)


Step 2500: loss = 2.1114
Step 5000: loss = 2.0764
Step 7500: loss = 2.1446
Step 10000: loss = 1.8986

Baseline (block 8, 5000 steps): train=1.9988, test=2.0719, gap=0.0731
A: block 8, 10000 steps: train=1.9101, test=2.0521, gap=0.1420


In [46]:
# Experiment B: same LayerNorm model, block_size 32 (4x context), 5000 steps.
# block_size is global: get_batch, position table, Head mask and generate all read it.
block_size = 32
try:
    torch.manual_seed(1337)
    ctx_model = LayerNormStackLanguageModel(vocab_size, n_embedd)
    print("Parameters:", sum(p.numel() for p in ctx_model.parameters()))
    context_losses = train_and_eval(ctx_model, 5000)
    print()
    report("Baseline (block 8, 5000 steps)", layernorm_losses)
    report("A: block 8, 10000 steps", longer_losses)
    report("B: block 32, 5000 steps", context_losses)

    m.eval()
    generated = m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
    print("\nSample (block 32):\n" + decode(generated[0].tolist()))
finally:
    block_size = 8  # restore so earlier cells keep working


Parameters: 39969
Step 2500: loss = 2.0696
Step 5000: loss = 1.8554

Baseline (block 8, 5000 steps): train=1.9988, test=2.0719, gap=0.0731
A: block 8, 10000 steps: train=1.9101, test=2.0521, gap=0.1420
B: block 32, 5000 steps: train=1.8451, test=1.9682, gap=0.1231

Sample (block 32):

We yead gand your this pain?

ANze Angett moulds. Kents I that'd woould head.

LUCIO:

THays peace an shalle my and pasting mer wear himall's me.
It Willy, now Luldsain the with eyess.
Alown hear and bany me, your soway takelds: and wortureat.

INLY ROLIO:
Ooods cace as of have go his tray. Wi, Ist I lay? I shalp, I well at of
a breiget I band morsian
Luch yis if mincancele cowearters,
Pence efor tumene two we Camard art mome finqhe sit;
Wert! Go musentrue; that mosce wayer, securs?
Apaide oplat


## Dropout
Randomly zeroes values during training only (`m.train()`); off in `m.eval()`. Applied to attention weights, multi-head output and feed-forward output.

In [47]:
# Dropout on one vector: train mode zeros ~p of values and scales the rest by 1/(1-p).
torch.manual_seed(0)
drop = nn.Dropout(p=0.5)
v = torch.ones(1, 8) * 2
drop.train()
print("train:", drop(v))
drop.eval()
print("eval: ", drop(v))


train: tensor([[0., 0., 4., 0., 0., 0., 4., 4.]])
eval:  tensor([[2., 2., 2., 2., 2., 2., 2., 2.]])


In [48]:
dropout = 0.2


class DropoutHead(Head):
    # Head with dropout on attention weights: randomly blocks some token-to-token links.
    def __init__(self, n_embedd, head_size):
        super().__init__(n_embedd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        _, T, _ = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        d_k = k.shape[-1]
        scores = q @ k.transpose(-2, -1) / d_k**0.5  # (B, T, T)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.dropout(F.softmax(scores, dim=-1))  # (B, T, T)
        return wei @ v  # (B, T, head_size)


class DropoutMultiHeadAttention(nn.Module):
    def __init__(self, n_embedd, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [DropoutHead(n_embedd, head_size) for _ in range(num_heads)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(torch.cat([h(x) for h in self.heads], dim=-1))


class DropoutBlock(LayerNormBlock):
    # LayerNorm block with dropout-enabled attention and feed-forward.
    def __init__(self, n_embedd, num_heads):
        super().__init__(n_embedd, num_heads)
        self.sa_heads = DropoutMultiHeadAttention(
            n_embedd, num_heads, n_embedd // num_heads
        )
        self.ffwd.net.append(nn.Dropout(dropout))  # after Linear 128 -> 32


class DropoutLanguageModel(LayerNormStackLanguageModel):
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer)
        self.blocks = nn.Sequential(
            *[DropoutBlock(n_embedd, num_heads) for _ in range(n_layer)],
            nn.LayerNorm(n_embedd),
        )


In [49]:
# Experiment C: experiment B (block 32, 5000 steps) plus dropout 0.2.
block_size = 32
try:
    torch.manual_seed(1337)
    drop_model = DropoutLanguageModel(vocab_size, n_embedd)
    print("Parameters:", sum(p.numel() for p in drop_model.parameters()))
    dropout_losses = train_and_eval(drop_model, 5000)
    print()
    report("B: block 32, no dropout", context_losses)
    report("C: block 32, dropout 0.2", dropout_losses)

    m.eval()  # dropout off for generation
    generated = m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
    print("\nSample (dropout):\n" + decode(generated[0].tolist()))
finally:
    block_size = 8


Parameters: 39969
Step 2500: loss = 2.2451
Step 5000: loss = 2.0890

B: block 32, no dropout: train=1.8451, test=1.9682, gap=0.1231
C: block 32, dropout 0.2: train=2.0027, test=2.0573, gap=0.0547

Sample (dropout):

Courd of he of sick, nefuse.

METRM:
I is hop if babe
Boma's he hichine;
Hedd previeder'ss of on art hee's
Frove, of my will for geencedsalf if her do thought cair the as
Tontays the uphatiing-as, you goo;
That wath mimp now.
Her and whilen am the in maker.
Bonem bady forgecus ast merter
To trey sherabing, and ha a my pran the two hath non I amm-be kneaguth

SINCARUKINGBES VINCES:
Peicergett wand If not fairghing mer,
Is that do your withe not ajever abe and the way of and not not of wall asholy


## Output projection
`Linear(32 → 32)` after `torch.cat`, so every output slot reads all heads before the result is added to `x`. Compared against experiment B (only change: proj).

In [50]:
class ProjMultiHeadAttention(nn.Module):
    # Heads -> cat -> proj (mixes all heads) -> dropout.
    def __init__(self, n_embedd, num_heads, dropout):
        super().__init__()
        self.heads = nn.ModuleList(
            [DropoutHead(n_embedd, n_embedd // num_heads) for _ in range(num_heads)]
        )
        for h in self.heads:
            h.dropout = nn.Dropout(dropout)  # per-model rate, not the global one
        self.proj = nn.Linear(n_embedd, n_embedd)  # 32 -> 32: every output reads all heads
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, 32), heads side by side
        out = self.proj(out)  # (B, T, 32), heads mixed
        return self.dropout(out)


class GPTBlock(LayerNormBlock):
    # Full GPT block: pre-norm, residuals, projection, dropout.
    def __init__(self, n_embedd, num_heads, dropout):
        super().__init__(n_embedd, num_heads)
        self.sa_heads = ProjMultiHeadAttention(n_embedd, num_heads, dropout)
        self.ffwd.net.append(nn.Dropout(dropout))


class GPTLanguageModel(LayerNormStackLanguageModel):
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3, dropout=0.0):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer)
        self.blocks = nn.Sequential(
            *[GPTBlock(n_embedd, num_heads, dropout) for _ in range(n_layer)],
            nn.LayerNorm(n_embedd),
        )


In [51]:
# Experiment D: experiment B (block 32, 5000 steps, no dropout) plus output projection.
block_size = 32
try:
    torch.manual_seed(1337)
    proj_model = GPTLanguageModel(vocab_size, n_embedd, dropout=0.0)
    print("Parameters:", sum(p.numel() for p in proj_model.parameters()))
    proj_losses = train_and_eval(proj_model, 5000)
    print()
    report("B: block 32", context_losses)
    report("C: block 32 + dropout 0.2", dropout_losses)
    report("D: block 32 + proj", proj_losses)

    m.eval()
    generated = m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)
    print("\nSample (proj):\n" + decode(generated[0].tolist()))
finally:
    block_size = 8


Parameters: 43137
Step 2500: loss = 2.0687
Step 5000: loss = 1.8665

B: block 32: train=1.8451, test=1.9682, gap=0.1231
C: block 32 + dropout 0.2: train=2.0027, test=2.0573, gap=0.0547
D: block 32 + proj: train=1.8410, test=1.9651, gap=0.1240

Sample (proj):

We yecou much nead thes. I wart he dight noouses.

PRITELING ELIAD: you heave?

CARCIO:
Hays and you nor you mysay neastingly,
The peartales! mose to bouy. Marcouses' mothes brooe;
Pooke fance of and ban, me, me seebl'd
Aand I harded beturia, age oucer; and offelcine, my ompainy tayisul!
No Eise, the lecing in on a the what knot breaten I butle,
As it with your in incame leave it of withen in for tume;
And mush mast had not, if happront;
We the U heer true; that mosior: yeater the thep;
he orst 


## Experiments: FFN activation
Same model as experiment D; only the activation between the two FFN `Linear` layers changes.

In [52]:
# What each activation does to the same values.
z = torch.tensor([-3.0, -1.0, -0.2, 0.0, 0.2, 1.0, 3.0])
print("z       :", z)
print("ReLU    :", nn.ReLU()(z))
print("GELU    :", nn.GELU()(z).round(decimals=3))
print("none    :", nn.Identity()(z))


z       : tensor([-3.0000, -1.0000, -0.2000,  0.0000,  0.2000,  1.0000,  3.0000])
ReLU    : tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.2000, 1.0000, 3.0000])
GELU    : tensor([-0.0040, -0.1590, -0.0840,  0.0000,  0.1160,  0.8410,  2.9960])
none    : tensor([-3.0000, -1.0000, -0.2000,  0.0000,  0.2000,  1.0000,  3.0000])


In [53]:
# Experiments E and F: experiment D with the FFN activation swapped.
def with_activation(model, act):
    # FFN net is [Linear, activation, Linear, Dropout]; replace index 1 in every block.
    for blk in model.blocks:
        if isinstance(blk, GPTBlock):
            blk.ffwd.net[1] = act()
    return model


block_size = 32
try:
    torch.manual_seed(1337)
    no_act_losses = train_and_eval(with_activation(GPTLanguageModel(vocab_size, n_embedd), nn.Identity), 5000)
    torch.manual_seed(1337)
    gelu_losses = train_and_eval(with_activation(GPTLanguageModel(vocab_size, n_embedd), nn.GELU), 5000)
    print()
    report("D: ReLU (current)", proj_losses)
    report("E: no activation", no_act_losses)
    report("F: GELU", gelu_losses)
finally:
    block_size = 8


Step 2500: loss = 2.2108
Step 5000: loss = 2.0176
Step 2500: loss = 2.0698
Step 5000: loss = 1.8194

D: ReLU (current): train=1.8410, test=1.9651, gap=0.1240
E: no activation: train=2.0077, test=2.0871, gap=0.0794
F: GELU: train=1.8172, test=1.9537, gap=0.1365


## Experiment: multi-query attention (MQA)
All heads keep their own query but share one key and one value projection. Only change vs experiment D.

In [54]:
class MultiQueryAttention(nn.Module):
    # Each head has its own query; all heads share ONE key and ONE value projection.
    def __init__(self, n_embedd, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = n_embedd // num_heads
        self.query = nn.Linear(n_embedd, n_embedd, bias=False)  # 4 heads x 8, fused then sliced
        self.key = nn.Linear(n_embedd, self.head_size, bias=False)  # shared: 32 -> 8
        self.value = nn.Linear(n_embedd, self.head_size, bias=False)  # shared: 32 -> 8
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.attn_dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(n_embedd, n_embedd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x).view(B, T, self.num_heads, self.head_size).transpose(1, 2)  # (B, 4, T, 8)
        k = self.key(x).unsqueeze(1)  # (B, 1, T, 8): broadcast to every head
        v = self.value(x).unsqueeze(1)  # (B, 1, T, 8)
        scores = q @ k.transpose(-2, -1) / self.head_size**0.5  # (B, 4, T, T)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.attn_dropout(F.softmax(scores, dim=-1))
        out = wei @ v  # (B, 4, T, 8)
        out = out.transpose(1, 2).reshape(B, T, C)  # same as torch.cat of heads: (B, T, 32)
        return self.dropout(self.proj(out))


class MQABlock(GPTBlock):
    def __init__(self, n_embedd, num_heads, dropout):
        super().__init__(n_embedd, num_heads, dropout)
        self.sa_heads = MultiQueryAttention(n_embedd, num_heads, dropout)


class MQALanguageModel(GPTLanguageModel):
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3, dropout=0.0):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer, dropout)
        self.blocks = nn.Sequential(
            *[MQABlock(n_embedd, num_heads, dropout) for _ in range(n_layer)],
            nn.LayerNorm(n_embedd),
        )


In [55]:
# Experiment G: experiment D (block 32, proj, ReLU, no dropout) with multi-query attention.
block_size = 32
try:
    torch.manual_seed(1337)
    mqa_model = MQALanguageModel(vocab_size, n_embedd)
    mha_params = sum(p.numel() for p in proj_model.parameters())
    mqa_params = sum(p.numel() for p in mqa_model.parameters())
    print(f"Parameters: MHA {mha_params:,} -> MQA {mqa_params:,} ({mqa_params - mha_params:,})")
    mqa_losses = train_and_eval(mqa_model, 5000)
    print()
    report("D: multi-head (4 K, 4 V)", proj_losses)
    report("G: multi-query (1 K, 1 V)", mqa_losses)

    # KV-cache size per token per block if we cached keys and values during generation.
    print(f"\nKV floats cached per token per block: MHA {2 * 4 * 8}, MQA {2 * 1 * 8}")
finally:
    block_size = 8


Parameters: MHA 43,137 -> MQA 38,529 (-4,608)
Step 2500: loss = 2.1102
Step 5000: loss = 1.8853

D: multi-head (4 K, 4 V): train=1.8410, test=1.9651, gap=0.1240
G: multi-query (1 K, 1 V): train=1.8814, test=1.9902, gap=0.1088

KV floats cached per token per block: MHA 64, MQA 16


## Experiment: grouped-query attention (GQA)
4 query heads, 2 K/V groups (each shared by 2 heads). Between multi-head (4 groups) and multi-query (1 group).

In [56]:
class GroupedQueryAttention(nn.Module):
    # num_kv_groups K/V projections, each shared by num_heads // num_kv_groups heads.
    # num_kv_groups = num_heads -> multi-head; num_kv_groups = 1 -> multi-query.
    def __init__(self, n_embedd, num_heads, num_kv_groups, dropout):
        super().__init__()
        assert num_heads % num_kv_groups == 0
        self.num_heads, self.num_kv_groups = num_heads, num_kv_groups
        self.head_size = n_embedd // num_heads
        self.query = nn.Linear(n_embedd, n_embedd, bias=False)  # 4 heads x 8
        self.key = nn.Linear(n_embedd, num_kv_groups * self.head_size, bias=False)  # 2 groups x 8
        self.value = nn.Linear(n_embedd, num_kv_groups * self.head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.attn_dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(n_embedd, n_embedd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        G, hs = self.num_kv_groups, self.head_size
        q = self.query(x).view(B, T, self.num_heads, hs).transpose(1, 2)  # (B, 4, T, 8)
        k = self.key(x).view(B, T, G, hs).transpose(1, 2)  # (B, 2, T, 8)
        v = self.value(x).view(B, T, G, hs).transpose(1, 2)  # (B, 2, T, 8)
        # Heads 0,1 use group 0; heads 2,3 use group 1.
        k = k.repeat_interleave(self.num_heads // G, dim=1)  # (B, 4, T, 8)
        v = v.repeat_interleave(self.num_heads // G, dim=1)  # (B, 4, T, 8)
        scores = q @ k.transpose(-2, -1) / hs**0.5  # (B, 4, T, T)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (wei @ v).transpose(1, 2).reshape(B, T, C)  # (B, T, 32)
        return self.dropout(self.proj(out))


class GQABlock(GPTBlock):
    def __init__(self, n_embedd, num_heads, num_kv_groups, dropout):
        super().__init__(n_embedd, num_heads, dropout)
        self.sa_heads = GroupedQueryAttention(n_embedd, num_heads, num_kv_groups, dropout)


class GQALanguageModel(GPTLanguageModel):
    def __init__(self, vocab_size, n_embedd, num_heads=4, n_layer=3, num_kv_groups=2, dropout=0.0):
        super().__init__(vocab_size, n_embedd, num_heads, n_layer, dropout)
        self.blocks = nn.Sequential(
            *[GQABlock(n_embedd, num_heads, num_kv_groups, dropout) for _ in range(n_layer)],
            nn.LayerNorm(n_embedd),
        )


In [57]:
# Experiment H: experiment D with grouped-query attention, 2 K/V groups for 4 heads.
block_size = 32
try:
    torch.manual_seed(1337)
    gqa_model = GQALanguageModel(vocab_size, n_embedd, num_kv_groups=2)
    print(f"Parameters: {sum(p.numel() for p in gqa_model.parameters()):,}")
    gqa_losses = train_and_eval(gqa_model, 5000)
    print()
    report("D: MHA, 4 K/V groups", proj_losses)
    report("H: GQA, 2 K/V groups", gqa_losses)
    report("G: MQA, 1 K/V group ", mqa_losses)
    print(f"\nKV floats cached per token per block: MHA {2*4*8}, GQA {2*2*8}, MQA {2*1*8}")
finally:
    block_size = 8


Parameters: 40,065
Step 2500: loss = 2.0755
Step 5000: loss = 1.8713

D: MHA, 4 K/V groups: train=1.8410, test=1.9651, gap=0.1240
H: GQA, 2 K/V groups: train=1.8560, test=1.9819, gap=0.1258
G: MQA, 1 K/V group : train=1.8814, test=1.9902, gap=0.1088

KV floats cached per token per block: MHA 64, GQA 32, MQA 16


## Experiment: BPE tokenizer instead of characters
Train a 1,000-token byte-pair encoding on the training split, then retrain run D's model on BPE tokens. Compare in bits per character, since a BPE token covers several characters.

In [61]:
# Train a small BPE tokenizer on the training split only (no peeking at test text).
import collections
import re

split_at = int(0.9 * len(text))
train_text, test_text = text[:split_at], text[split_at:]

# Pre-split into chunks: words keep their leading space; punctuation and whitespace stand alone.
# Merges never cross chunk boundaries, so " the" can become a token but "e k" cannot.
CHUNK = re.compile(r" ?[A-Za-z']+| ?[^A-Za-z'\s]|\s")
chunk_counts = collections.Counter(CHUNK.findall(train_text))

bpe_vocab = {i: c for i, c in enumerate(chars)}  # start: the 65 characters
merges = {}  # (id_a, id_b) -> new id, in the order learned
pieces = {w: [stoi[c] for c in w] for w in chunk_counts}
bpe_vocab_size = 1000

while len(bpe_vocab) < bpe_vocab_size:
    # Count every adjacent pair, weighted by how often the chunk appears.
    pairs = collections.Counter()
    for w, ids in pieces.items():
        for pair in zip(ids, ids[1:]):
            pairs[pair] += chunk_counts[w]
    (a, b), _ = pairs.most_common(1)[0]
    new_id = len(bpe_vocab)
    merges[a, b] = new_id
    bpe_vocab[new_id] = bpe_vocab[a] + bpe_vocab[b]
    # Replace that pair everywhere with the new token.
    for w, ids in pieces.items():
        out, i = [], 0
        while i < len(ids):
            if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                out.append(new_id)
                i += 2
            else:
                out.append(ids[i])
                i += 1
        pieces[w] = out

print("First 20 merges:", [bpe_vocab[i] for i in range(65, 85)])
print("Last 20 merges: ", [bpe_vocab[i] for i in range(980, 1000)])


First 20 merges: [' t', 'he', ' a', 'ou', ' s', ' m', 'in', ' w', 're', 'ha', ' the', 'nd', ' b', 'is', 'or', ' f', 'er', 'll', 'it', 'on']
Last 20 merges:  [' seem', ' daughter', 'ried', 'UCKING', 'UCKINGHAM', ' first', ' tre', ' thine', 'ix', 'BUCKINGHAM', 'ESS', ' BOL', ' BOLING', ' BOLINGBR', ' BOLINGBRO', ' BOLINGBROKE', 'LAU', ' though', 'men', ' bra']


In [62]:
# Encode / decode with the learned merges.
_chunk_cache = {}


def bpe_encode_chunk(chunk):
    if chunk not in _chunk_cache:
        ids = [stoi[c] for c in chunk]
        while len(ids) > 1:
            # Apply the earliest-learned merge available in this chunk.
            pair = min(zip(ids, ids[1:]), key=lambda pr: merges.get(pr, float("inf")))
            if pair not in merges:
                break
            out, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
                    out.append(merges[pair])
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        _chunk_cache[chunk] = ids
    return _chunk_cache[chunk]


def bpe_encode(s):
    return [t for chunk in CHUNK.findall(s) for t in bpe_encode_chunk(chunk)]


def bpe_decode(tokens):
    return "".join(bpe_vocab[t] for t in tokens)


bpe_train = torch.tensor(bpe_encode(train_text), dtype=torch.long)
bpe_test = torch.tensor(bpe_encode(test_text), dtype=torch.long)
assert bpe_decode(bpe_test.tolist()) == test_text

sample = "KING RICHARD II:\nWhat said the matter?"
ids = bpe_encode(sample)
print("Pieces:", [bpe_vocab[t] for t in ids])
print(f"{len(sample)} chars -> {len(ids)} tokens")
chars_per_token = len(test_text) / len(bpe_test)
print(f"Test split: {len(test_text):,} chars -> {len(bpe_test):,} tokens ({chars_per_token:.2f} chars/token)")


Pieces: ['KING', ' RICHARD', ' II', ':', '\n', 'What', ' sa', 'id', ' the', ' m', 'atter', '?']
38 chars -> 12 tokens
Test split: 111,540 chars -> 47,905 tokens (2.33 chars/token)


In [60]:
# Experiment I: run D's model with BPE tokens instead of characters.
# Loss per token != loss per char, so compare bits per character (bpc) on the same test text.
import math


def bpc(loss_per_token, n_tokens, n_chars):
    return loss_per_token * n_tokens / n_chars / math.log(2)


char_data = (train_data, test_data)
train_data, test_data = bpe_train, bpe_test  # get_batch reads these globals
try:
    results = {}
    for bs in (32, 13):  # 32 tokens ~ 81 chars; 13 tokens ~ 32 chars (same text span as run D)
        block_size = bs
        torch.manual_seed(1337)
        bpe_model = GPTLanguageModel(bpe_vocab_size, n_embedd)
        print(f"block {bs}: parameters {sum(p.numel() for p in bpe_model.parameters()):,}")
        results[bs] = train_and_eval(bpe_model, 5000)
        if bs == 32:
            m.eval()
            sample_ids = m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=200)
            bpe_sample = bpe_decode(sample_ids[0].tolist())
finally:
    train_data, test_data = char_data
    block_size = 8

n_test_chars, n_test_tokens = len(test_text), len(bpe_test)
print(f"\n{'run':38s} {'test loss':>10s} {'test bpc':>9s}")
print(f"{'D: chars, block 32 (32 chars)':38s} {proj_losses['test']:10.4f} {proj_losses['test'] / math.log(2):9.4f}")
for bs, span in ((13, "~32 chars"), (32, "~81 chars")):
    loss = results[bs]["test"]
    print(f"{f'I: BPE-1000, block {bs} ({span})':38s} {loss:10.4f} {bpc(loss, n_test_tokens, n_test_chars):9.4f}")
print("\nSample (BPE, block 32):\n" + bpe_sample)


block 32: parameters 103,912
Step 2500: loss = 4.0807
Step 5000: loss = 3.8882
block 13: parameters 103,304
Step 2500: loss = 4.3992
Step 5000: loss = 3.8016

run                                     test loss  test bpc
D: chars, block 32 (32 chars)              1.9651    2.8350
I: BPE-1000, block 13 (~32 chars)          4.0788    2.5273
I: BPE-1000, block 32 (~81 chars)          4.0126    2.4863

Sample (BPE, block 32):


So have war same that show'd.

LADY AN OFUMOW:
To your honourable, my seated to full free
As first place stay of burn'd us that answerse. The time.

Gentleman:
We says, he, sir.

QUEEN:
Now is the cause by thy fass:
On so was hand, away is better

Thhost the Gir as tricks in death-
Shall hither some late forthsel bew'd and farewell.
I did hear you be true--beelargher to come with them.

Nurse:
Thou lost thou have makerk the world fe of their to; to:
In day! oples Hastide their depr'd and face
thly as I have been so?


## References

Experiment summaries, including MHA/GQA/MQA results and cache accounting: [Experiment results](https://github.com/cerdo03/llm-systems-exercises/blob/main/docs/experiment-results.md#multi-head-vs-grouped-query-vs-multi-query).

Source code and tests: [cerdo03/llm-systems-exercises](https://github.com/cerdo03/llm-systems-exercises).
